# Question Answering Pipeline

**Authors:** Roberto Herrera and Bach Nguyen  
**Datasets:** HotPotQA and QASPER

We implement a retrieval-augmented extractive question answering pipeline. Stage 1 selects relevant text through chunking, embedding, and ChromaDB retrieval, implementing the assignment’s summarization stage. Stage 2 uses an extractive QA model to predict an answer from the retrieved context.

## 1. Setup

We define dataset locations, load the pipeline dependencies, and create a results directory. Experiment settings and outputs are saved with each run.

In [5]:
import os
import re
import ast
import json
import random
from datetime import datetime
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_huggingface import HuggingFaceEmbeddings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

HOTPOTQA_PATH = Path(
    "/root/.cache/kagglehub/datasets/"
    "jeromeblanchet/hotpotqa-question-answering-dataset/"
    "versions/1/hotpot_train_v1.1.json"
)

# change when running on different machine
MACHINE_LABEL = "chameleonTACC"
RESULTS_DIR = Path("/app/results") / MACHINE_LABEL
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


## 2. Load datasets
HotPotQA provides candidate Wikipedia paragraphs for each question.
QASPER groups questions and answer annotations by research paper.

In [6]:
import ast
import json
import re

import pandas as pd

with HOTPOTQA_PATH.open(encoding="utf-8") as f:
    hotpot = json.load(f)

print("HotPotQA questions:", len(hotpot))

HotPotQA questions: 90447


## 3. Chunking

We compare recursive and semantic chunking. Recursive splitting uses a token-counted size cap, while semantic splitting identifies boundaries using a percentile threshold on adjacent text-unit embedding distances. Oversized sentences and semantic groups are split to respect the configured cap. Both methods use the active embedding tokenizer and preserve source metadata and character offsets.

In [7]:
import numpy as np
import spacy
from langchain_text_splitters import RecursiveCharacterTextSplitter

nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")

def make_splitter(chunk_size, chunk_overlap, tokenizer, token_limit):
    # chunk_size counts the special tokens the embedding model adds
    if not 0 < chunk_size <= token_limit:
        raise ValueError("chunk_size is larger than the embedding model allows")

    n_special = tokenizer.num_special_tokens_to_add(pair=False)
    max_tokens = chunk_size - n_special  # room left for the text itself

    if not 0 <= chunk_overlap < max_tokens:
        raise ValueError("chunk_overlap must be smaller than the chunk size")

    def count_tokens(text):
        return len(tokenizer.encode(text, add_special_tokens=False, truncation=False))

    return RecursiveCharacterTextSplitter(
        chunk_size=max_tokens,
        chunk_overlap=chunk_overlap,
        length_function=count_tokens,
        add_start_index=True,
    )


def semantic_chunks(text, metadata, embedder, splitter, percentile):
    if not 0 < percentile < 100:
        raise ValueError("percentile must be between 0 and 100")

    # split into sentences, and split any sentence that is longer than the cap
    units = [] 
    for sent in nlp(text).sents:
        if not sent.text.strip():
            continue
        for piece in splitter.create_documents([sent.text]):
            start = sent.start_char + piece.metadata["start_index"]
            units.append((piece.page_content, start))

    if not units:
        return []

    # cosine distance between each unit and the next one
    breaks = []
    if len(units) > 1:
        vectors = np.array(embedder.embed_documents([u[0] for u in units]), dtype=float)
        vectors /= np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12)
        sims = np.sum(vectors[:-1] * vectors[1:], axis=1)
        distances = 1 - np.clip(sims, -1, 1)
        cutoff = np.percentile(distances, percentile)
        breaks = [i + 1 for i, d in enumerate(distances) if d > cutoff]

    docs = []
    first = 0
    for last in breaks + [len(units)]:
        start = units[first][1]
        end = units[last - 1][1] + len(units[last - 1][0])
        pieces = splitter.create_documents([text[start:end]], metadatas=[metadata])
        for piece in pieces:
            piece.metadata["start_index"] += start
            docs.append(piece)
        first = last

    return docs

## 4. Store chunks in ChromaDB

Each chunk is embedded and stored with its source identifier and character range.

HotPotQA uses a separate collection for each question’s candidate paragraphs. QASPER reuses a paper’s collection across its questions. Collections use cosine distance to compare question and chunk embeddings.

In [8]:
from time import perf_counter
from uuid import uuid4

import torch
from langchain_chroma import Chroma


def hotpot_paragraphs(example):
    # each HotPotQA question comes with about ten (title, sentences) paragraphs
    return [
        (" ".join(sentences), {"title": title, "para_id": pid})
        for pid, (title, sentences) in enumerate(example["context"])
    ]


def build_store(paragraphs, embedder, chunking, chunk_size,
                chunk_overlap=0, percentile=None, timings=None):
    def now():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return perf_counter()

    t0 = now()

    if chunking not in ("recursive", "semantic"):
        raise ValueError("chunking must be 'recursive' or 'semantic'")
    if chunking == "semantic":
        if percentile is None or not 0 < percentile < 100:
            raise ValueError("semantic chunking needs a percentile between 0 and 100")
        if chunk_overlap != 0:
            raise ValueError("semantic chunking only works with zero overlap")

    tokenizer = embedder._client.tokenizer
    splitter = make_splitter(
        chunk_size, chunk_overlap, tokenizer, embedder._client.max_seq_length
    )

    # cut every paragraph into chunks and remember where each chunk came from
    docs = []
    for text, metadata in paragraphs:
        if chunking == "recursive":
            pieces = splitter.create_documents([text], metadatas=[metadata])
        else:
            pieces = semantic_chunks(text, metadata, embedder, splitter, percentile)

        for piece in pieces:
            start = piece.metadata["start_index"]
            end = start + len(piece.page_content)

            # a chunk must be an exact slice of its paragraph and fit the cap
            if start < 0 or text[start:end] != piece.page_content:
                raise ValueError("chunk does not match the paragraph text")
            n_tokens = len(tokenizer.encode(
                piece.page_content, add_special_tokens=True, truncation=False
            ))
            if n_tokens > chunk_size:
                raise ValueError("chunk is longer than chunk_size")

            piece.metadata.update({
                "end_index": end,
                "chunk_id": f"{metadata['para_id']}:{start}:{end}",
                "embedding_tokens": n_tokens,
            })
            docs.append(piece)

    t1 = now()
    if timings is not None:
        timings["chunking_s"] = t1 - t0
        timings["embedding_store_s"] = 0.0

    # embed the chunks and put them in ChromaDB with cosine distance
    store = None
    if docs:
        store = Chroma.from_documents(
            documents=docs,
            embedding=embedder,
            ids=[doc.metadata["chunk_id"] for doc in docs],
            collection_name="retrieval_" + uuid4().hex,
            collection_metadata={"hnsw:space": "cosine"},
        )

    t2 = now()
    if timings is not None:
        if docs:
            timings["embedding_store_s"] = t2 - t1
        timings["build_total_s"] = t2 - t0

    return store

## 5. Retrieve relevant chunks
Single-pass retrieval searches using the original question. Two-hop retrieval uses the first retrieved chunk to expand the question, then searches again for additional chunks. Both return up to k unique chunks. We retain cosine distances and search-stage information to examine whether the second search adds useful evidence. Distances from different searches are not treated as one combined ranking.

In [9]:
from time import perf_counter

import torch

def retrieve(store, question, k, method, query_tokenizer,
             max_query_tokens, timings=None):
    def now():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return perf_counter()

    started = now()

    if k < 1:
        raise ValueError("k must be at least 1")
    if method not in ("single_pass", "two_hop"):
        raise ValueError("method must be 'single_pass' or 'two_hop'")
    if store is None:
        if timings is not None:
            timings["retrieval_s"] = now() - started
        return {"chunks": [], "expanded_query": None}

    def token_count(text):
        return len(query_tokenizer.encode(text, add_special_tokens=True))

    if token_count(question) > max_query_tokens:
        raise ValueError("question is longer than the embedding model allows")

    first = store.similarity_search_with_score(
        question, k=k if method == "single_pass" else 1
    )
    selected = [(doc, distance, 1) for doc, distance in first]
    expanded_query = None

    if method == "two_hop" and k > 1 and first:
        evidence = first[0][0].page_content
        offsets = query_tokenizer(
            evidence, add_special_tokens=False,
            return_offsets_mapping=True, truncation=False,
        )["offset_mapping"]

        end = min(len(offsets), max_query_tokens - token_count(question))
        while end > 0:
            candidate = question + "\n" + evidence[:offsets[end - 1][1]]
            if token_count(candidate) <= max_query_tokens:
                expanded_query = candidate
                break
            end -= 1

        if expanded_query is None:
            raise ValueError("no first hop evidence fits next to the question")

        skip_first_paragraph = {"para_id": {"$ne": first[0][0].metadata["para_id"]}}
        seen = {doc.metadata["chunk_id"] for doc, _, _ in selected}
        second = store.similarity_search_with_score(
            expanded_query, k=k, filter=skip_first_paragraph
        )

        for doc, distance in second:
            chunk_id = doc.metadata["chunk_id"]
            if chunk_id not in seen:
                selected.append((doc, distance, 2))
                seen.add(chunk_id)
            if len(selected) == k:
                break

    result = {
        "chunks": [
            {"rank": rank, "text": doc.page_content, "metadata": dict(doc.metadata),
             "distance": float(distance), "hop": hop}
            for rank, (doc, distance, hop) in enumerate(selected, start=1)
        ],
        "expanded_query": expanded_query,
    }

    if timings is not None:
        timings["retrieval_s"] = now() - started

    return result

## 6. Prepare the QA context

Retrieved chunks are added in retrieval order within the reader’s input budget, including the question and special tokens. If a chunk does not fit, only its fitting prefix is included and packing stops. We record the final context, token count, and retained source ranges to distinguish retrieval failures from evidence lost during packing.

In [10]:
from time import perf_counter

def pack_context(question, chunks, qa_tokenizer,
                 max_tokens=512, timings=None):
    started = perf_counter()

    def input_length(context):
        return len(qa_tokenizer(
            question, context, add_special_tokens=True,
            truncation=False,
        )["input_ids"])

    if input_length("") >= max_tokens:
        raise ValueError("The question leaves no room for QA context.")

    context = ""
    included = []

    for chunk in chunks:
        text = chunk["text"]
        separator = "\n\n" if context else ""
        candidate = context + separator + text
        truncated = False

        if input_length(candidate) > max_tokens:
            offsets = qa_tokenizer(
                text, add_special_tokens=False,
                return_offsets_mapping=True, truncation=False,
            )["offset_mapping"]
            end = min(len(offsets), max_tokens)

            while end > 0:
                prefix = text[:offsets[end - 1][1]]
                candidate = context + separator + prefix
                if input_length(candidate) <= max_tokens:
                    text = prefix
                    break
                end -= 1

            if end == 0:
                break
            truncated = True

        context_start = len(context) + len(separator)
        context = candidate
        metadata = dict(chunk["metadata"])
        metadata["end_index"] = metadata["start_index"] + len(text)

        included.append({
            "rank": chunk["rank"],
            "text": text,
            "metadata": metadata,
            "context_start": context_start,
            "context_end": len(context),
            "truncated": truncated,
        })

        if truncated:
            break

    result = {
        "context": context,
        "input_tokens": input_length(context),
        "included_chunks": included,
    }

    if timings is not None:
        timings["packing_s"] = perf_counter() - started

    return result

## 7. Question answering

We compare three pretrained extractive QA models. Each reader receives the question and packed context and returns an answer span with a confidence score. Model loading and inference are timed separately. Abstention is configurable and recorded for each experiment.

In [36]:
from time import perf_counter

import torch
from transformers import pipeline

QA_MODELS = [
    "deepset/tinyroberta-squad2",
    "deepset/roberta-base-squad2",
    "google-bert/bert-large-uncased-whole-word-masking-finetuned-squad",
    "distilbert/distilbert-base-cased-distilled-squad",
]

QA_TOKEN_LIMIT = 512
MAX_ANSWER_LEN = 512


def synced_time():
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    return perf_counter()


def load_qa_model(model_name):
    started = synced_time()
    qa_model = pipeline(
        "question-answering",
        model=model_name,
        tokenizer=model_name,
        device=0 if DEVICE == "cuda" else -1,
    )
    qa_model.model.eval()
    return qa_model, synced_time() - started


def answer_question(qa_model, question, packed, timings, allow_empty):
    actual_tokens = len(qa_model.tokenizer(
        question,
        packed["context"],
        add_special_tokens=True,
        truncation=False,
    )["input_ids"])

    if actual_tokens > QA_TOKEN_LIMIT:
        raise ValueError("Packed input exceeds the QA token limit.")
    if actual_tokens != packed["input_tokens"]:
        raise ValueError("Pack the context with this QA model's tokenizer.")

    started = synced_time()

    if packed["context"].strip():
        prediction = qa_model(
            question=question,
            context=packed["context"],
            max_seq_len=QA_TOKEN_LIMIT,
            max_question_len=QA_TOKEN_LIMIT,
            max_answer_len=MAX_ANSWER_LEN,
            doc_stride=0,
            handle_impossible_answer=allow_empty,
        )
    else:
        prediction = {
            "answer": "",
            "score": 0.0,
            "start": 0,
            "end": 0,
        }

    timings["qa_s"] = synced_time() - started

    if prediction["answer"]:
        extracted = packed["context"][
            prediction["start"]:prediction["end"]
        ]
        if extracted != prediction["answer"]:
            raise ValueError("Predicted answer does not match its context span.")

    return prediction

## 8. Answer evaluation

We report normalized exact match and word-token F1 against the best matching reference answer. QASPER annotations are preserved as alternatives; multiple spans within one annotation are concatenated. Answer types are retained for separate reporting because an extractive reader cannot represent every answer type equally well. These are transparent baseline metrics rather than a reproduction of QASPER’s official evaluator.

In [12]:
import re
import string
from collections import Counter


def normalize_answer(text):
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def score_answer(prediction, references):
    if not references:
        return {"exact_match": None, "f1": None}

    predicted = normalize_answer(prediction)
    scores = []

    for reference in references:
        gold = normalize_answer(reference)
        p, g = predicted.split(), gold.split()
        overlap = sum((Counter(p) & Counter(g)).values())

        if not p or not g:
            f1 = float(p == g)
        else:
            f1 = 2 * overlap / (len(p) + len(g))

        scores.append((float(predicted == gold), f1))

    return {
        "exact_match": max(em for em, _ in scores),
        "f1": max(f1 for _, f1 in scores),
    }


references = {}

for example in hotpot:
    answer = example["answer"]
    references[("hotpot", str(example["_id"]), str(example["_id"]))] = {
        "answers": [answer],
        "answer_type": "yes_no" if answer.lower() in {"yes", "no"} else "span",
    }

## 9. Slices
The questions are filtered and permuted with a fixed seed, then split into a development slice of 300, a test slice of 1,000 and a load pool. The IDs and the seed are saved to `/app/results/slices.json`.


In [13]:
import hashlib
import json
from pathlib import Path

import numpy as np

SLICES_PATH = Path("/app/results/slices.json")

if SLICES_PATH.exists():
    print("Already saved:", json.loads(SLICES_PATH.read_text())["counts"])
else:
    kept = []
    for example in hotpot:
        answer = example["answer"]
        if answer.lower() in {"yes", "no"}:
            continue
        target = normalize_answer(answer)
        if target and any(
            target in normalize_answer(" ".join(sentences))
            for _, sentences in example["context"]
        ):
            kept.append(str(example["_id"]))

    order = np.random.default_rng(8001).permutation(len(kept))
    shuffled = [kept[i] for i in order]
    slices = {
        "dev": shuffled[:300],
        "test": shuffled[300:1300],
        "load": shuffled[1300:],
    }

    digest = hashlib.sha256()
    with HOTPOTQA_PATH.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)

    SLICES_PATH.write_text(json.dumps({
        "dataset": "HotpotQA distractor train, Kaggle jeromeblanchet mirror",
        "kaggle_version": 1,
        "file": HOTPOTQA_PATH.name,
        "file_sha256": digest.hexdigest(),
        "numpy_seed": 8001,
        "total_questions": len(hotpot),
        "filtered_questions": len(kept),
        "counts": {name: len(ids) for name, ids in slices.items()},
        "slices": slices,
    }))
    print("Total:", len(hotpot), "| filtered:", len(kept))
    print({name: len(ids) for name, ids in slices.items()})

Already saved: {'dev': 300, 'test': 1000, 'load': 83658}


## 10. Evaluation helpers

We report normalized answer exact match and token F1, together with source-paragraph and sentence-level evidence precision, recall and F1 before and after context packing. Sentence coverage requires the complete sentence to be present across retained source ranges. QASPER annotation alternatives are preserved, and evidence that cannot be fully mapped to the available text is reported separately. Results include attempted, scored and failed evaluations, actual input lengths, and stage latency distributions.

In [41]:
import json
import os
import platform
import random
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch

K_REPORT = (1, 3, 5, 10)
SLICES_PATH = Path("/app/results/slices.json")


def clean_text(text):
    return " ".join(text.lower().split())


def make_units(split, n_hotpot=None):
    ids = json.loads(SLICES_PATH.read_text())["slices"][split]
    if n_hotpot is not None:
        ids = ids[:n_hotpot]

    by_id = {str(example["_id"]): example for example in hotpot}
    units = {"hotpot": []}

    for example in (by_id[question_id] for question_id in ids):
        spans = {}
        title_to_pid = {}

        for pid, (title, sentences) in enumerate(example["context"]):
            title_to_pid[title] = pid
            position, items = 0, []

            for sentence in sentences:
                left = len(sentence) - len(sentence.lstrip())
                right = len(sentence.rstrip())
                items.append((position + left, position + right))
                position += len(sentence) + 1

            spans[pid] = items

        gold_sentences = set()
        mapped = True

        for title, idx in example["supporting_facts"]:
            pid = title_to_pid.get(title)
            if pid is None or not 0 <= idx < len(spans[pid]):
                mapped = False
            else:
                gold_sentences.add((pid, idx))

        evidence_sets = []
        if mapped and gold_sentences:
            evidence_sets.append({
                "docs": {pid for pid, _ in gold_sentences},
                "sentences": gold_sentences,
            })

        units["hotpot"].append({
            "context_id": str(example["_id"]),
            "paragraphs": hotpot_paragraphs(example),
            "sentence_spans": spans,
            "questions": [{
                "question_id": str(example["_id"]),
                "text": example["question"],
                "evidence_sets": evidence_sets,
                "evidence_status": "ok" if evidence_sets else "unmapped_evidence",
                "unmapped_evidence_items": (
                    0 if mapped else len(example["supporting_facts"])
                ),
            }],
        })

    return units


def beginning_chunks(unit):
    return {
        "chunks": [
            {
                "rank": rank,
                "text": text,
                "metadata": {
                    **metadata,
                    "start_index": 0,
                    "end_index": len(text),
                    "chunk_id": f"{metadata['para_id']}:0:{len(text)}",
                },
                "distance": None,
                "hop": 0,
            }
            for rank, (text, metadata) in enumerate(unit["paragraphs"], start=1)
        ],
        "expanded_query": None,
    }


def prf(gold, found):
    if not gold:
        return None, None, None
    hits = len(gold & found)
    precision = hits / len(found) if found else 0.0
    recall = hits / len(gold)
    f1 = 2 * hits / (len(gold) + len(found))
    return precision, recall, f1


def covered_sentences(chunks, spans):
    ranges = {}
    for chunk in chunks:
        metadata = chunk["metadata"]
        ranges.setdefault(metadata["para_id"], []).append(
            (metadata["start_index"], metadata["end_index"])
        )

    covered = set()
    for pid, intervals in ranges.items():
        merged = []
        for start, end in sorted(intervals):
            if merged and start <= merged[-1][1]:
                merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
            else:
                merged.append((start, end))

        for idx, (start, end) in enumerate(spans[pid]):
            if end > start and any(a <= start and end <= b for a, b in merged):
                covered.add((pid, idx))

    return covered


EVIDENCE_COLUMNS = [
    f"{stage}_{level}_{metric}"
    for stage in ("retrieved", "window")
    for level in ("doc", "sent")
    for metric in ("precision", "recall", "f1")
]


def evidence_metrics(unit, question, chunks, stage):
    out = {name: None for name in EVIDENCE_COLUMNS if name.startswith(stage)}
    alternatives = question["evidence_sets"]
    if not alternatives:
        return out

    found_docs = {c["metadata"]["para_id"] for c in chunks}
    found_sentences = covered_sentences(chunks, unit["sentence_spans"])

    for level, found in (("doc", found_docs), ("sent", found_sentences)):
        key = "docs" if level == "doc" else "sentences"
        scores = [prf(alt[key], found) for alt in alternatives]
        # Keep precision and recall from the same best-F1 annotation.
        p, r, f = max(scores, key=lambda values: values[2])
        out.update({
            f"{stage}_{level}_precision": p,
            f"{stage}_{level}_recall": r,
            f"{stage}_{level}_f1": f,
        })

    return out


def machine_info():
    return {
        "label": MACHINE_LABEL,
        "platform": platform.platform(),
        "python": platform.python_version(),
        "logical_processors": os.cpu_count(),
        "device": DEVICE,
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    }


KEYS = ["dataset", "embedder", "chunking", "cap", "search", "k", "qa"]


def summarize(df):
    if df.empty:
        return pd.DataFrame()

    data = df.copy()
    data["failed"] = data["status"].ne("ok")
    data["succeeded"] = ~data["failed"]

    for column in EVIDENCE_COLUMNS:
        if column not in data:
            data[column] = np.nan

    timings = [
        "build_total_s", "build_amortized_s", "retrieval_s",
        "packing_s", "qa_s", "qa_stage_s", "pipeline_stage_sum_s",
    ]
    for column in timings + ["input_tokens", "n_chunks"]:
        if column not in data:
            data[column] = np.nan

    aggregations = {
        "questions": ("status", "size"),
        "successful_questions": ("succeeded", "sum"),
        "errors": ("failed", "sum"),
        "scored_questions": ("exact_match", "count"),
        "exact_match": ("exact_match", "mean"),
        "f1": ("f1", "mean"),
        "confidence_passes": ("confidence_pass", "sum"),
        "mean_input_tokens": ("input_tokens", "mean"),
        "mean_packed_chunks": ("n_chunks", "mean"),
    }

    for column in EVIDENCE_COLUMNS:
        aggregations[column] = (column, "mean")
        aggregations[column + "_n"] = (column, "count")

    for column in timings:
        aggregations[column + "_mean"] = (column, "mean")

    aggregations["pipeline_stage_sum_s_p95"] = (
        "pipeline_stage_sum_s", lambda values: values.quantile(0.95)
    )

    return data.groupby(KEYS, dropna=False).agg(
        **aggregations
    ).reset_index()

## 11. Experiment runner
Each context collection is built once per embedding and chunking configuration, and retrieval outputs are saved for reuse across readers. QA models are loaded one at a time and evaluated at each requested retrieval depth. Every record includes the question, reference answers, retrieved chunks, retained context, evidence metrics, prediction and stage timings. Construction cost is reported separately. Retrieval plus reader-stage time is a sum of separately measured stages, not a directly measured live request latency. Completed retrieval blocks and QA records can be reused when resuming the same run directory.

In [15]:
import gc
import json
import traceback
from datetime import datetime
from pathlib import Path
from time import perf_counter

import pandas as pd
import torch
from langchain_huggingface import HuggingFaceEmbeddings

CONFIDENCE_THRESHOLD = 0.5

def memory_snapshot():
    snapshot = {}
    try:
        import resource
        snapshot["peak_rss_mb"] = (
            resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
        )
    except ImportError:
        pass
    if torch.cuda.is_available():
        snapshot["cuda_peak_allocated_mb"] = (
            torch.cuda.max_memory_allocated() / 2**20
        )
    return snapshot
    
def run_experiment(name, units, embedders, chunkings, caps, searches, qa_models,
                   ks=K_REPORT, percentile=90, run_dir=None):
    ks = sorted(set(ks))
    if not ks or any(k < 1 for k in ks):
        raise ValueError("Provide positive retrieval depths.")
    if not set(searches) <= {"single_pass", "two_hop", "beginning"}:
        raise ValueError("Unknown search method.")
    if not qa_models:
        raise ValueError("Provide at least one QA model.")

    if run_dir is None:
        run_dir = RESULTS_DIR / f"{name}_{datetime.now():%Y%m%d_%H%M%S_%f}"
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    cache_dir = run_dir / "retrieval"
    cache_dir.mkdir(exist_ok=True)

    config = {
        "name": name,
        "embedders": list(embedders),
        "chunkings": list(chunkings),
        "caps": list(caps),
        "searches": list(searches),
        "qa_models": list(qa_models),
        "ks": ks,
        "percentile": percentile,
        "chunk_overlap": 0,
        "qa_token_limit": QA_TOKEN_LIMIT,
        "max_answer_len": MAX_ANSWER_LEN,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "seed": SEED,
        "timing_mode": "build_plus_saved_retrieval_plus_measured_qa_stage",
        "sample_ids": {
            dataset: [
                [unit["context_id"],
                 [q["question_id"] for q in unit["questions"]]]
                for unit in dataset_units
            ]
            for dataset, dataset_units in units.items()
        },
    }

    config_path = run_dir / "config.json"
    if config_path.exists():
        if json.loads(config_path.read_text()) != config:
            raise ValueError("Resume settings differ. Use a new run directory.")
    else:
        config_path.write_text(json.dumps(config, indent=2))
        (run_dir / "machine.json").write_text(
            json.dumps(machine_info(), indent=2)
        )

    lookup = {
        (dataset, unit["context_id"], question["question_id"]): (unit, question)
        for dataset, dataset_units in units.items()
        for unit in dataset_units
        for question in unit["questions"]
    }

    def cache_record(dataset, unit, question, embedder_name,
                     chunking, cap, search, k, build_times,
                     retrieved=None, retrieval_s=None, error=None):
        refs = references.get(
            (dataset, unit["context_id"], question["question_id"]),
            {"answers": [], "answer_type": "missing_reference"},
        )
        record = {
            "dataset": dataset,
            "context_id": unit["context_id"],
            "question_id": question["question_id"],
            "question": question["text"],
            "embedder": embedder_name,
            "chunking": chunking,
            "cap": cap,
            "search": search,
            "k": k,
            "answer_type": refs["answer_type"],
            "reference_answers": refs["answers"],
            "evidence_status": question["evidence_status"],
            "unmapped_evidence_items": question["unmapped_evidence_items"],
            "empty_context": not bool(unit["paragraphs"]),
            "build_total_s": build_times.get("build_total_s"),
            "chunking_s": build_times.get("chunking_s"),
            "embedding_store_s": build_times.get("embedding_store_s"),
            "build_amortized_s": (
                build_times["build_total_s"] / len(unit["questions"])
                if "build_total_s" in build_times else None
            ),
            "retrieval_s": retrieval_s,
            "retrieved": retrieved,
            "retrieval_error": error,
        }
        if retrieved is not None:
            record.update(evidence_metrics(
                unit, question, retrieved["chunks"], "retrieved"
            ))
        return record

    cache_paths = []
    retrieval_methods = [s for s in searches if s != "beginning"]

    # Run the beginning-of-context baseline once, independently of embedders.
    if "beginning" in searches:
        path = cache_dir / "beginning.jsonl"
        cache_paths.append(path)

        if not path.exists():
            temporary = path.with_suffix(".partial")
            with temporary.open("w", encoding="utf-8") as output:
                for dataset, dataset_units in units.items():
                    for unit in dataset_units:
                        for question in unit["questions"]:
                            record = cache_record(
                                dataset, unit, question, "none", "none", None,
                                "beginning", None,
                                {"build_total_s": 0.0, "chunking_s": 0.0,
                                 "embedding_store_s": 0.0},
                                beginning_chunks(unit), 0.0,
                            )
                            output.write(json.dumps(record, ensure_ascii=False) + "\n")
            temporary.replace(path)

    model_log = run_dir / "model_loads.jsonl"
    started_run = perf_counter()

    for embedder_name in embedders if retrieval_methods else []:
        model_id = (
            embedder_name if "/" in embedder_name
            else "sentence-transformers/" + embedder_name
        )
        started = synced_time()
        embedder = HuggingFaceEmbeddings(
            model_name=model_id,
            model_kwargs={"device": DEVICE},
        )
        load_s = synced_time() - started
        embedder.embed_query("Warm-up query.")
        token_limit = embedder._client.max_seq_length
        tokenizer = embedder._client.tokenizer

        with model_log.open("a", encoding="utf-8") as output:
            output.write(json.dumps({
                "stage": "embedding", "model": model_id,
                "load_s": load_s, "token_limit": token_limit,
            }) + "\n")

        for chunking in chunkings:
            for cap in caps:
                if cap > token_limit:
                    print(f"Skipped: {model_id}, cap {cap} > {token_limit}")
                    continue

                block_id = (
                    f"{model_id.replace('/', '__')}_{chunking}_{cap}"
                )
                path = cache_dir / f"{block_id}.jsonl"
                cache_paths.append(path)

                if path.exists():
                    print("Reusing:", path.name, flush=True)
                    continue

                temporary = path.with_suffix(".partial")
                count = 0

                with temporary.open("w", encoding="utf-8") as output:
                    for dataset, dataset_units in units.items():
                        for unit in dataset_units:
                            if not unit["questions"]:
                                continue

                            store, build_error, build_times = None, None, {}
                            try:
                                try:
                                    store = build_store(
                                        unit["paragraphs"], embedder, chunking,
                                        cap, 0,
                                        percentile if chunking == "semantic" else None,
                                        build_times,
                                    )
                                except Exception:
                                    build_error = traceback.format_exc()

                                for question in unit["questions"]:
                                    for search in retrieval_methods:
                                        for k in ks:
                                            retrieved, error, times = None, build_error, {}
                                            if error is None:
                                                try:
                                                    retrieved = retrieve(
                                                        store, question["text"], k,
                                                        search, tokenizer, token_limit,
                                                        times,
                                                    )
                                                except Exception:
                                                    error = traceback.format_exc()

                                            record = cache_record(
                                                dataset, unit, question, model_id,
                                                chunking, cap, search, k,
                                                build_times, retrieved,
                                                times.get("retrieval_s"), error,
                                            )
                                            output.write(
                                                json.dumps(record, ensure_ascii=False) + "\n"
                                            )
                                            count += 1

                                    output.flush()
                                    if count % 100 == 0:
                                        print(block_id, count, "retrieval records",
                                              flush=True)
                            finally:
                                if store is not None:
                                    store.delete_collection()

                temporary.replace(path)
                print("Saved:", path.name, flush=True)
        
        with model_log.open("a", encoding="utf-8") as output:
            output.write(json.dumps({
                "stage": "embedding_done", "model": model_id,
                **memory_snapshot(),
            }) + "\n")
        del tokenizer, embedder
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    def record_key(record):
        return (
            record["dataset"], record["context_id"], record["question_id"],
            record["embedder"], record["chunking"], record["cap"],
            record["search"], record["k"], record["qa"],
        )

    predictions_path = run_dir / "records.jsonl"
    completed = set()

    # Remove only an incomplete trailing record left by an interruption.
    if predictions_path.exists():
        with predictions_path.open("rb+") as source:
            while True:
                position = source.tell()
                line = source.readline()
                if not line:
                    break
                try:
                    saved = json.loads(line)
                except json.JSONDecodeError:
                    if source.read():
                        raise ValueError("Corrupt record inside records.jsonl.")
                    source.seek(position)
                    source.truncate()
                    break
                completed.add(record_key(saved))

    gc.collect()

    with predictions_path.open("a", encoding="utf-8") as output:
        for qa_name in qa_models:
            qa_model, load_s = load_qa_model(qa_name)
            qa_model(question="What is it?", context="It is a test.",
                     max_seq_len=QA_TOKEN_LIMIT)

            with model_log.open("a", encoding="utf-8") as log:
                log.write(json.dumps({
                    "stage": "qa", "model": qa_name, "load_s": load_s,
                }) + "\n")

            for path in cache_paths:
                with path.open(encoding="utf-8") as source:
                    for line in source:
                        saved = json.loads(line)
                        record = {**saved, "qa": qa_name}
                        key = record_key(record)
                        if key in completed:
                            continue

                        unit, question = lookup[(
                            record["dataset"], record["context_id"],
                            record["question_id"],
                        )]
                        allow_empty = False
                        record.update({
                            "status": "ok",
                            "allow_empty": allow_empty,
                            "confidence_pass": False,
                        })

                        try:
                            if record["retrieval_error"]:
                                raise RuntimeError(record["retrieval_error"])

                            times = {}
                            started = synced_time()
                            packed = pack_context(
                                question["text"], record["retrieved"]["chunks"],
                                qa_model.tokenizer, QA_TOKEN_LIMIT, times,
                            )
                            prediction = answer_question(
                                qa_model, question["text"], packed, times,
                                allow_empty,
                            )
                            qa_stage_s = synced_time() - started

                            record.update({
                                "prediction": prediction,
                                "packed": packed,
                                "confidence": float(prediction["score"]),
                                "confidence_pass": (
                                    bool(packed["context"].strip())
                                    and prediction["score"] >= CONFIDENCE_THRESHOLD
                                ),
                                "packing_s": times["packing_s"],
                                "qa_s": times["qa_s"],
                                "qa_stage_s": qa_stage_s,
                                "pipeline_stage_sum_s": (
                                    record["build_total_s"]
                                    + record["retrieval_s"] + qa_stage_s
                                ),
                                "n_chunks": len(packed["included_chunks"]),
                                "input_tokens": packed["input_tokens"],
                                "truncated_last": any(
                                    c["truncated"] for c in packed["included_chunks"]
                                ),
                                **evidence_metrics(
                                    unit, question, packed["included_chunks"], "window"
                                ),
                                **score_answer(
                                    prediction["answer"], record["reference_answers"]
                                ),
                            })
                        except Exception:
                            has_refs = bool(record["reference_answers"])
                            record.update({
                                "status": "error",
                                "error": traceback.format_exc(),
                                "exact_match": 0.0 if has_refs else None,
                                "f1": 0.0 if has_refs else None,
                            })

                        output.write(json.dumps(record, ensure_ascii=False) + "\n")
                        output.flush()
                        completed.add(key)

                        if len(completed) % 100 == 0:
                            print(len(completed), "QA records saved", flush=True)
            with model_log.open("a", encoding="utf-8") as log:
                log.write(json.dumps({
                    "stage": "qa_done", "model": qa_name,
                    **memory_snapshot(),
                }) + "\n")
            del qa_model
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

    excluded_fields = {
        "retrieved", "packed", "prediction", "reference_answers",
        "question", "error", "retrieval_error",
    }
    rows = []
    with predictions_path.open(encoding="utf-8") as source:
        for line in source:
            record = json.loads(line)
            rows.append({
                key: value for key, value in record.items()
                if key not in excluded_fields
            })

    df = pd.DataFrame(rows)
    df.to_csv(run_dir / "question_metrics.csv", index=False)
    summary = summarize(df)
    summary.to_csv(run_dir / "summary.csv", index=False)

    print("Finished in", round((perf_counter() - started_run) / 60, 1), "minutes")
    print("Saved:", run_dir)
    return df, summary

## 12. Experiments
All experiments run from one plan. Each run is saved in its own folder, and a run whose folder already holds finished results for the same settings and the same questions is reused. The combined table covers every configuration.

In [42]:
from time import perf_counter

# True runs a small sample. False runs all 300 dev questions.
QUICK = False

SIZES = {"n_hotpot": 3} if QUICK else {"n_hotpot": 300}
FINAL_DIR = RESULTS_DIR / ("quick" if QUICK else "dev")
FINAL_DIR.mkdir(parents=True, exist_ok=True)

L6 = ["sentence-transformers/all-MiniLM-L6-v2"]
L12_MPNET = [
    "sentence-transformers/all-MiniLM-L12-v2",
    "sentence-transformers/all-mpnet-base-v2",
]
BGE_SMALL = ["BAAI/bge-small-en-v1.5"]
TINY = ["deepset/tinyroberta-squad2"]
SEARCHES = ["single_pass", "two_hop"]
KS = (1, 3, 5, 10)

PLAN = [
    dict(name="baseline", embedders=L6, chunkings=["recursive"], caps=[128],
         searches=["beginning"], qa_models=TINY, ks=KS),
    dict(name="recursive", embedders=L6, chunkings=["recursive"], caps=[128],
         searches=SEARCHES, qa_models=TINY, ks=KS),
    dict(name="semantic", embedders=L6, chunkings=["semantic"], caps=[128],
         searches=SEARCHES, qa_models=TINY, ks=KS),
    dict(name="chunk_size", embedders=L6, chunkings=["recursive"], caps=[64, 256],
         searches=SEARCHES, qa_models=TINY, ks=KS),
    dict(name="embedders", embedders=L12_MPNET, chunkings=["recursive"], caps=[128],
         searches=SEARCHES, qa_models=TINY, ks=KS),
    dict(name="bge_small", embedders=BGE_SMALL, chunkings=["recursive"], caps=[128],
         searches=SEARCHES, qa_models=TINY, ks=KS),
    dict(name="qa_models", embedders=L6, chunkings=["recursive"], caps=[128],
         searches=SEARCHES, qa_models=QA_MODELS[1:], ks=KS),
]

units = make_units("dev", **SIZES)
n_questions = sum(len(u["questions"]) for group in units.values() for u in group)
print("Questions:", n_questions, "| quick:", QUICK)

started = perf_counter()
summaries = []
for run in PLAN:
    print("\nRun:", run["name"], flush=True)
    _, summary = run_experiment(
        run["name"], units, run_dir=FINAL_DIR / run["name"],
        **{key: value for key, value in run.items() if key != "name"},
    )
    summaries.append(summary)

combined = pd.concat(summaries, ignore_index=True)
combined.to_csv(FINAL_DIR / "combined_summary.csv", index=False)
print("\nEvaluations:", int(combined["questions"].sum()),
      "| errors:", int(combined["errors"].sum()),
      "| minutes:", round((perf_counter() - started) / 60, 1))

Questions: 300 | quick: False

Run: baseline


Device set to use cpu


Finished in 0.0 minutes
Saved: /app/results/chameleonTACC/dev/baseline

Run: recursive
Reusing: sentence-transformers__all-MiniLM-L6-v2_recursive_128.jsonl


Device set to use cpu


Finished in 0.1 minutes
Saved: /app/results/chameleonTACC/dev/recursive

Run: semantic
Reusing: sentence-transformers__all-MiniLM-L6-v2_semantic_128.jsonl


Device set to use cpu


... (5 outputs left out)

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (613 > 512). Running this sequence through the model will result in indexing errors


100 QA records saved
200 QA records saved
300 QA records saved
400 QA records saved
500 QA records saved
600 QA records saved
700 QA records saved
800 QA records saved
900 QA records saved
1000 QA records saved
1100 QA records saved
1200 QA records saved
1300 QA records saved
1400 QA records saved
1500 QA records saved
1600 QA records saved
1700 QA records saved
1800 QA records saved
1900 QA records saved
2000 QA records saved
2100 QA records saved
2200 QA records saved
2300 QA records saved
2400 QA records saved
Finished in 4.8 minutes
Saved: /app/results/chameleonTACC/dev/bge_small

Run: qa_models
Reusing: sentence-transformers__all-MiniLM-L6-v2_recursive_128.jsonl


Device set to use cpu
Some weights of the model checkpoint at google-bert/bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (552 > 512). Running this sequence through the model will result in indexing errors


5300 QA records saved
5400 QA records saved
5500 QA records saved
5600 QA records saved
5700 QA records saved
5800 QA records saved
5900 QA records saved
6000 QA records saved
6100 QA records saved
6200 QA records saved
6300 QA records saved
6400 QA records saved
6500 QA records saved
6600 QA records saved
6700 QA records saved
6800 QA records saved
6900 QA records saved
7000 QA records saved
7100 QA records saved
7200 QA records saved
Finished in 2.2 minutes
Saved: /app/results/chameleonTACC/dev/qa_models

Evaluations: 24300 | errors: 0 | minutes: 7.3


## 13. Assertions
The development records are used to pick one confidence threshold for each QA model. The thresholds are saved to `thresholds.json` in the results folder, and the pass rate of each assertion is printed for each QA model.

In [43]:
import json
import re
from pathlib import Path

import pandas as pd

DEV_DIR = RESULTS_DIR / "dev"
THRESHOLDS = [round(0.05 * i, 2) for i in range(1, 20)]
MONTHS = set(
    "january february march april may june july august "
    "september october november december".split()
)
NUMBER_WORDS = set(
    "one two three four five six seven eight nine ten eleven twelve "
    "thirteen fourteen fifteen sixteen seventeen eighteen nineteen "
    "twenty thirty forty fifty sixty seventy eighty ninety hundred "
    "thousand million billion dozen".split()
)


def load_records(directory):
    rows = []
    for path in sorted(Path(directory).glob("*/records.jsonl")):
        with path.open(encoding="utf-8") as source:
            for line in source:
                record = json.loads(line)
                if record["status"] != "ok":
                    continue
                rows.append({
                    "run": path.parent.name,
                    "question_id": record["question_id"],
                    "question": record["question"],
                    "embedder": record["embedder"],
                    "chunking": record["chunking"],
                    "cap": record["cap"],
                    "search": record["search"],
                    "k": record["k"],
                    "qa": record["qa"],
                    "answer": record["prediction"]["answer"],
                    "score": record["confidence"],
                    "f1": record["f1"],
                })
    return pd.DataFrame(rows)


def answer_type_ok(question, answer):
    q = question.lower().strip()
    words = set(re.findall(r"[a-z]+", answer.lower()))
    if q.startswith(("when", "what year", "in what year")):
        return bool(re.search(r"\b\d{4}\b", answer)) or bool(words & MONTHS)
    if q.startswith(("how many", "how much")):
        return bool(re.search(r"\d", answer)) or bool(words & NUMBER_WORDS)
    return True


def tune_threshold(group):
    correct = group["f1"] >= 0.5
    best_t, best_score = None, -1
    for t in THRESHOLDS:
        passed = group["score"] >= t
        tpr = (passed & correct).sum() / max(correct.sum(), 1)
        tnr = (~passed & ~correct).sum() / max((~correct).sum(), 1)
        balanced = (tpr + tnr) / 2
        if balanced > best_score:
            best_t, best_score = t, balanced
    return best_t, best_score


def add_assertions(df, thresholds):
    df = df.copy()
    df["A1"] = df["answer"].str.strip().ne("")
    df["A2"] = df["answer"].str.split().str.len().le(15)
    df["A3"] = [answer_type_ok(q, a) for q, a in zip(df["question"], df["answer"])]
    df["A4"] = df["score"] >= df["qa"].map(thresholds)
    df["all_pass"] = df[["A1", "A2", "A3", "A4"]].all(axis=1)
    return df


dev = load_records(DEV_DIR)

thresholds = {}
for qa, group in dev.groupby("qa"):
    t, balanced = tune_threshold(group)
    thresholds[qa] = t
    print(qa, "| t =", t, "| balanced accuracy =", round(balanced, 3),
          "| records =", len(group))

(RESULTS_DIR / "thresholds.json").write_text(json.dumps(thresholds, indent=2))

dev = add_assertions(dev, thresholds)
print(dev.groupby("qa")[["A1", "A2", "A3", "A4", "all_pass"]].mean().round(3))

deepset/roberta-base-squad2 | t = 0.15 | balanced accuracy = 0.661 | records = 2400
deepset/tinyroberta-squad2 | t = 0.1 | balanced accuracy = 0.615 | records = 17100
distilbert/distilbert-base-cased-distilled-squad | t = 0.55 | balanced accuracy = 0.653 | records = 2400
google-bert/bert-large-uncased-whole-word-masking-finetuned-squad | t = 0.55 | balanced accuracy = 0.696 | records = 2400
                                                       A1     A2     A3  \
qa                                                                        
deepset/roberta-base-squad2                         0.998  0.949  0.983   
deepset/tinyroberta-squad2                          1.000  0.942  0.987   
distilbert/distilbert-base-cased-distilled-squad    1.000  0.944  0.988   
google-bert/bert-large-uncased-whole-word-maski...  1.000  0.964  0.984   

                                                       A4  all_pass  
qa                                                                   
deepset/roberta

## 14. Best retrieval depth

For each configuration, the value of k with the highest F1 on the development questions is selected. The test questions are run at this k.

In [44]:
import pandas as pd

DEV_DIR = RESULTS_DIR / "dev"
GROUP_KEYS = ["chunking", "embedder", "cap", "search", "qa"]
SHOW = GROUP_KEYS + ["k", "f1", "exact_match", "window_sent_recall",
                     "pipeline_stage_sum_s_p95"]

dev_summary = pd.concat(
    [pd.read_csv(path) for path in sorted(DEV_DIR.glob("*/summary.csv"))],
    ignore_index=True,
)
dev_summary["embedder"] = dev_summary["embedder"].str.split("/").str[-1]
dev_summary["qa"] = dev_summary["qa"].str.split("/").str[-1]

best_k = dev_summary.loc[
    dev_summary.groupby(GROUP_KEYS, dropna=False)["f1"].idxmax()
]
print(best_k[SHOW].round(3).to_string(index=False))

 chunking          embedder   cap      search                                                    qa    k    f1  exact_match  window_sent_recall  pipeline_stage_sum_s_p95
     none              none   NaN   beginning                                    tinyroberta-squad2  NaN 0.273        0.197               0.416                     0.331
recursive all-MiniLM-L12-v2 128.0 single_pass                                    tinyroberta-squad2  5.0 0.484        0.400               0.780                     2.503
recursive all-MiniLM-L12-v2 128.0     two_hop                                    tinyroberta-squad2  3.0 0.529        0.447               0.727                     2.149
recursive  all-MiniLM-L6-v2  64.0 single_pass                                    tinyroberta-squad2  5.0 0.487        0.397               0.572                     1.125
recursive  all-MiniLM-L6-v2  64.0     two_hop                                    tinyroberta-squad2  5.0 0.495        0.417               0.589       

## 15. Test run

Each of the 15 configurations is run once on the 1,000 test questions, at the k selected on the development questions. Configurations that share an embedder, chunking, chunk size and k are run together.

In [45]:
from pathlib import Path
from time import perf_counter

import pandas as pd

# True runs a small sample. False runs all 1,000 test questions.
QUICK = False

SIZES = {"n_hotpot": 3} if QUICK else {"n_hotpot": 1000}
DEV = RESULTS_DIR / "dev"
FINAL_DIR = RESULTS_DIR / ("quick_test" if QUICK else "test")
FINAL_DIR.mkdir(parents=True, exist_ok=True)

# Best k for each configuration, from the development questions
GROUP_KEYS = ["chunking", "embedder", "cap", "search", "qa"]
dev_metrics = pd.concat(
    [pd.read_csv(path) for path in sorted(DEV.glob("*/question_metrics.csv"))],
    ignore_index=True,
)
embedder_ids = {e.split("/")[-1]: e for e in dev_metrics["embedder"].dropna().unique()}
dev_metrics["embedder"] = dev_metrics["embedder"].str.split("/").str[-1]
dev_metrics["qa"] = dev_metrics["qa"].str.split("/").str[-1]
dev_f1 = (
    dev_metrics.groupby(GROUP_KEYS + ["k"], dropna=False)["f1"]
    .mean().reset_index()
)
best_k = dev_f1.loc[dev_f1.groupby(GROUP_KEYS, dropna=False)["f1"].idxmax()]

qa_ids = {model.split("/")[-1]: model for model in QA_MODELS}
L6 = "sentence-transformers/all-MiniLM-L6-v2"
TINY_NAME = "tinyroberta-squad2"

PLAN = []

baseline = best_k[best_k["search"] == "beginning"]
for _, row in baseline.iterrows():
    PLAN.append(dict(
        name="baseline", embedders=[L6], chunkings=["recursive"], caps=[128],
        searches=["beginning"], qa_models=[qa_ids[row["qa"]]], ks=(1,),
    ))

# tinyroberta configurations that share an embedder, chunking, cap and k run together
groups = {}
others = []
for _, row in best_k[best_k["search"] != "beginning"].iterrows():
    if row["qa"] == TINY_NAME:
        key = (row["embedder"], row["chunking"], int(row["cap"]), int(row["k"]))
        groups.setdefault(key, []).append(row["search"])
    else:
        others.append(row)

for (embedder, chunking, cap, k), searches in sorted(groups.items()):
    PLAN.append(dict(
        name=f"{chunking}_{embedder}_cap{cap}_k{k}",
        embedders=[embedder_ids[embedder]], chunkings=[chunking], caps=[cap],
        searches=sorted(set(searches)), qa_models=[qa_ids[TINY_NAME]], ks=(k,),
    ))

# the other QA models run once per search at their own best k
for row in others:
    cap, k = int(row["cap"]), int(row["k"])
    PLAN.append(dict(
        name=f"{row['chunking']}_{row['embedder']}_cap{cap}_{row['search']}_k{k}_{row['qa']}",
        embedders=[embedder_ids[row["embedder"]]], chunkings=[row["chunking"]],
        caps=[cap], searches=[row["search"]], qa_models=[qa_ids[row["qa"]]], ks=(k,),
    ))

print("Configurations:", len(best_k), "| runs:", len(PLAN))

units = make_units("test", **SIZES)
n_questions = sum(len(u["questions"]) for group in units.values() for u in group)
print("Questions:", n_questions, "| quick:", QUICK)

started = perf_counter()
summaries = []
for run in PLAN:
    print("\nRun:", run["name"], flush=True)
    _, summary = run_experiment(
        run["name"], units, run_dir=FINAL_DIR / run["name"],
        **{key: value for key, value in run.items() if key != "name"},
    )
    summaries.append(summary)

combined = pd.concat(summaries, ignore_index=True)
combined.to_csv(FINAL_DIR / "combined_summary.csv", index=False)
print("\nEvaluations:", int(combined["questions"].sum()),
      "| errors:", int(combined["errors"].sum()),
      "| minutes:", round((perf_counter() - started) / 60, 1))

Configurations: 21 | runs: 15
Questions: 1000 | quick: False

Run: baseline


Device set to use cpu


Finished in 0.0 minutes
Saved: /app/results/chameleonTACC/test/baseline

Run: recursive_all-MiniLM-L12-v2_cap128_k3
Reusing: sentence-transformers__all-MiniLM-L12-v2_recursive_128.jsonl


Device set to use cpu


Finished in 0.1 minutes
Saved: /app/results/chameleonTACC/test/recursive_all-MiniLM-L12-v2_cap128_k3

Run: recursive_all-MiniLM-L12-v2_cap128_k5
Reusing: sentence-transformers__all-MiniLM-L12-v2_recursive_128.jsonl


Device set to use cpu


... (21 outputs left out)

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (585 > 512). Running this sequence through the model will result in indexing errors


100 QA records saved
200 QA records saved
300 QA records saved
400 QA records saved
500 QA records saved
600 QA records saved
700 QA records saved
800 QA records saved
900 QA records saved
1000 QA records saved
Finished in 5.5 minutes
Saved: /app/results/chameleonTACC/test/recursive_all-MiniLM-L6-v2_cap128_two_hop_k10_distilbert-base-cased-distilled-squad

Run: recursive_all-MiniLM-L6-v2_cap128_two_hop_k10_roberta-base-squad2
sentence-transformers__all-MiniLM-L6-v2_recursive_128 100 retrieval records
sentence-transformers__all-MiniLM-L6-v2_recursive_128 200 retrieval records
sentence-transformers__all-MiniLM-L6-v2_recursive_128 300 retrieval records
sentence-transformers__all-MiniLM-L6-v2_recursive_128 400 retrieval records
sentence-transformers__all-MiniLM-L6-v2_recursive_128 500 retrieval records
sentence-transformers__all-MiniLM-L6-v2_recursive_128 600 retrieval records
sentence-transformers__all-MiniLM-L6-v2_recursive_128 700 retrieval records
sentence-transformers__all-MiniLM-L6-v

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (603 > 512). Running this sequence through the model will result in indexing errors


100 QA records saved
200 QA records saved
300 QA records saved
400 QA records saved
500 QA records saved
600 QA records saved
700 QA records saved
800 QA records saved
900 QA records saved
1000 QA records saved
Finished in 6.1 minutes
Saved: /app/results/chameleonTACC/test/recursive_all-MiniLM-L6-v2_cap128_two_hop_k10_roberta-base-squad2

Evaluations: 21000 | errors: 2 | minutes: 45.9


## 16. Test results

For each configuration, the test questions give the F1, exact match, answer in window rate, supporting sentence recall and the pass rate of each assertion. A question that failed to run counts as wrong.

In [46]:
import json
import re
from pathlib import Path

import pandas as pd

TEST_DIR = RESULTS_DIR / "test"
thresholds = json.loads((RESULTS_DIR / "thresholds.json").read_text())

MONTHS = set(
    "january february march april may june july august "
    "september october november december".split()
)
NUMBER_WORDS = set(
    "one two three four five six seven eight nine ten eleven twelve "
    "thirteen fourteen fifteen sixteen seventeen eighteen nineteen "
    "twenty thirty forty fifty sixty seventy eighty ninety hundred "
    "thousand million billion dozen".split()
)


def answer_type_ok(question, answer):
    q = question.lower().strip()
    words = set(re.findall(r"[a-z]+", answer.lower()))
    if q.startswith(("when", "what year", "in what year")):
        return bool(re.search(r"\b\d{4}\b", answer)) or bool(words & MONTHS)
    if q.startswith(("how many", "how much")):
        return bool(re.search(r"\d", answer)) or bool(words & NUMBER_WORDS)
    return True


def answer_in_window(record):
    context = normalize_answer(record["packed"]["context"])
    return any(
        normalize_answer(ref) and normalize_answer(ref) in context
        for ref in record["reference_answers"]
    )


rows = []
for path in sorted(TEST_DIR.glob("*/records.jsonl")):
    with path.open(encoding="utf-8") as source:
        for line in source:
            r = json.loads(line)
            ok = r["status"] == "ok"
            row = {
                "chunking": r["chunking"],
                "embedder": r["embedder"].split("/")[-1],
                "cap": r["cap"],
                "search": r["search"],
                "qa": r["qa"].split("/")[-1],
                "k": r["k"],
                "error": not ok,
                "f1": r["f1"],
                "exact_match": r["exact_match"],
                "window_sent_recall": r.get("window_sent_recall") if ok else 0.0,
                "in_window": False, "A1": False, "A2": False,
                "A3": False, "A4": False,
            }
            if ok:
                answer = r["prediction"]["answer"]
                row["in_window"] = answer_in_window(r)
                row["A1"] = answer.strip() != ""
                row["A2"] = len(answer.split()) <= 15
                row["A3"] = answer_type_ok(r["question"], answer)
                row["A4"] = r["confidence"] >= thresholds[r["qa"]]
            rows.append(row)

test = pd.DataFrame(rows)
test["all_pass"] = test[["A1", "A2", "A3", "A4"]].all(axis=1)

REPORT_KEYS = ["chunking", "embedder", "cap", "search", "qa"]
table = test.groupby(REPORT_KEYS, dropna=False).agg(
    k=("k", "first"),
    questions=("f1", "size"),
    errors=("error", "sum"),
    f1=("f1", "mean"),
    exact_match=("exact_match", "mean"),
    in_window=("in_window", "mean"),
    window_sent_recall=("window_sent_recall", "mean"),
    A1=("A1", "mean"), A2=("A2", "mean"), A3=("A3", "mean"),
    A4=("A4", "mean"), all_pass=("all_pass", "mean"),
).reset_index().sort_values("f1", ascending=False)

table.to_csv(TEST_DIR / "test_report.csv", index=False)
print(table.round(3).to_string(index=False))

 chunking          embedder   cap      search                                                    qa    k  questions  errors    f1  exact_match  in_window  window_sent_recall    A1    A2    A3    A4  all_pass
recursive  all-MiniLM-L6-v2 128.0     two_hop bert-large-uncased-whole-word-masking-finetuned-squad 10.0       1000       0 0.588        0.518      0.849               0.811 1.000 0.946 0.994 0.489     0.487
recursive  all-MiniLM-L6-v2 128.0 single_pass bert-large-uncased-whole-word-masking-finetuned-squad 10.0       1000       0 0.571        0.500      0.805               0.792 1.000 0.949 0.992 0.488     0.485
recursive bge-small-en-v1.5 128.0     two_hop                                    tinyroberta-squad2  3.0       1000       0 0.533        0.452      0.840               0.796 1.000 0.947 0.998 0.526     0.516
recursive bge-small-en-v1.5 128.0 single_pass                                    tinyroberta-squad2  3.0       1000       0 0.514        0.431      0.816               

## 17.Assertion pass rate against F1

The Pearson and Spearman correlations between F1 and the pass rate are computed over all configurations and over the configurations that use tinyroberta. The ranks of both are listed side by side.

In [47]:
import pandas as pd

TEST_DIR = RESULTS_DIR / "test"
report = pd.read_csv(TEST_DIR / "test_report.csv")

is_base = report["search"].eq("beginning")
is_tiny = report["qa"].eq("tinyroberta-squad2")

sets = {
    "all with baseline": report,
    "all without baseline": report[~is_base],
    "tinyroberta with baseline": report[is_tiny],
    "tinyroberta without baseline": report[is_tiny & ~is_base],
}

rows = []
for name, part in sets.items():
    for column in ["A4", "all_pass"]:
        rows.append({
            "set": name,
            "configurations": len(part),
            "pass rate": column,
            "pearson": round(part["f1"].corr(part[column]), 3),
            "spearman": round(part["f1"].corr(part[column], method="spearman"), 3),
        })
print(pd.DataFrame(rows).to_string(index=False))

ranked = report.assign(
    f1_rank=report["f1"].rank(ascending=False, method="min").astype(int),
    all_pass_rank=report["all_pass"].rank(ascending=False, method="min").astype(int),
).sort_values("f1_rank")
print()
print(ranked[["qa", "embedder", "cap", "search", "f1", "all_pass",
              "f1_rank", "all_pass_rank"]].round(3).to_string(index=False))

                         set  configurations pass rate  pearson  spearman
           all with baseline              21        A4    0.673     0.350
           all with baseline              21  all_pass    0.680     0.372
        all without baseline              20        A4    0.322     0.247
        all without baseline              20  all_pass    0.345     0.272
   tinyroberta with baseline              15        A4    0.885     0.617
   tinyroberta with baseline              15  all_pass    0.883     0.627
tinyroberta without baseline              14        A4    0.658     0.528
tinyroberta without baseline              14  all_pass    0.650     0.541

                                                   qa          embedder   cap      search    f1  all_pass  f1_rank  all_pass_rank
bert-large-uncased-whole-word-masking-finetuned-squad  all-MiniLM-L6-v2 128.0     two_hop 0.588     0.487        1              8
bert-large-uncased-whole-word-masking-finetuned-squad  all-MiniLM-L6-v2 1

## Comparison with Milestone 1 accuracy

In Milestone 1, accuracy was the share of questions whose confidence passed the threshold, without checking whether the answer was correct. Here an answer is correct when its F1 is at least 0.5, and the threshold for each QA model is chosen on the development questions to maximize balanced accuracy. The Milestone 1 accuracy is computed for every configuration on the test slice and placed next to F1 and the all assertions pass rate. The correlation between F1 and the old accuracy is printed at the bottom.

In [48]:
import json
import pandas as pd

TEST_DIR = RESULTS_DIR / "test"
config_columns = ["chunking", "embedder", "cap", "search", "qa"]

rows = []
for path in sorted(TEST_DIR.glob("*/records.jsonl")):
    with path.open(encoding="utf-8") as source:
        for line in source:
            r = json.loads(line)
            rows.append({
                "chunking": r["chunking"],
                "embedder": r["embedder"].split("/")[-1],
                "cap": r["cap"],
                "search": r["search"],
                "qa": r["qa"].split("/")[-1],
                "old_accuracy": bool(r["confidence_pass"]),
            })

old = (
    pd.DataFrame(rows)
    .groupby(config_columns, dropna=False)["old_accuracy"].mean()
    .reset_index()
)
report = pd.read_csv(TEST_DIR / "test_report.csv")
both = report.merge(old, on=config_columns, how="left")
both = both[config_columns + ["f1", "old_accuracy", "all_pass"]]
print(both.sort_values("f1", ascending=False).round(3).to_string(index=False))

print()
print("F1 against old accuracy, Pearson", round(both["f1"].corr(both["old_accuracy"]), 3),
      "| Spearman", round(both["f1"].corr(both["old_accuracy"], method="spearman"), 3))

 chunking          embedder   cap      search                                                    qa    f1  old_accuracy  all_pass
recursive  all-MiniLM-L6-v2 128.0     two_hop bert-large-uncased-whole-word-masking-finetuned-squad 0.588         0.523     0.487
recursive  all-MiniLM-L6-v2 128.0 single_pass bert-large-uncased-whole-word-masking-finetuned-squad 0.571         0.516     0.485
recursive bge-small-en-v1.5 128.0     two_hop                                    tinyroberta-squad2 0.533         0.339     0.516
recursive bge-small-en-v1.5 128.0 single_pass                                    tinyroberta-squad2 0.514         0.337     0.531
recursive  all-MiniLM-L6-v2 128.0     two_hop                                   roberta-base-squad2 0.503         0.207     0.447
recursive all-mpnet-base-v2 128.0     two_hop                                    tinyroberta-squad2 0.490         0.332     0.496
recursive  all-MiniLM-L6-v2 128.0 single_pass                                   roberta-ba

## Model registry

Each variant of the two scalable stages is stored in MinIO with its name, version, source, size on disk and test F1. The embedding variants use recursive chunking at 128 and are scored with tinyroberta. The QA model variants are scored with all-MiniLM-L6-v2. The version is the commit of the model repository, and F1 is given for single pass and two hop on the 1,000 test questions.

In [51]:
import io
import json
import os
import shutil
from pathlib import Path

import pandas as pd
from huggingface_hub import HfApi, snapshot_download
from minio import Minio

ENDPOINT = os.environ.get("MINIO_ENDPOINT", "172.17.0.1:9000")
BUCKET = "models"
CACHE_DIR = Path("/tmp/registry_models")

EMBEDDING_VARIANTS = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-MiniLM-L12-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "BAAI/bge-small-en-v1.5",
]
SKIP = ["*.h5", "*.msgpack", "*.onnx", "*.ot", "*.tflite", "*.tar.gz",
        "tf_model*", "flax_model*", "rust_model*", "openvino_model*",
        "onnx/*", "openvino/*", "coreml/*"]

report = pd.read_csv(RESULTS_DIR / "test" / "test_report.csv")
base = report[report["chunking"].eq("recursive") & report["cap"].eq(128)]


def test_f1(embedder, qa):
    rows = base[base["embedder"].eq(embedder) & base["qa"].eq(qa)]
    scores = rows.set_index("search")["f1"]
    return round(scores.get("single_pass", float("nan")), 3), \
        round(scores.get("two_hop", float("nan")), 3)


client = Minio(
    ENDPOINT,
    access_key=os.environ.get("MINIO_USER", "qa_registry"),
    secret_key=os.environ.get("MINIO_PASSWORD", "qa_registry_pass"),
    secure=False,
)
if not client.bucket_exists(BUCKET):
    client.make_bucket(BUCKET)

variants = [("embedding", repo) for repo in EMBEDDING_VARIANTS]
variants += [("qa", repo) for repo in QA_MODELS]

registry = []
for stage, repo in variants:
    name = repo.split("/")[-1]
    version = HfApi().model_info(repo).sha
    folder = CACHE_DIR / stage / name
    shutil.rmtree(folder, ignore_errors=True)
    snapshot_download(repo, revision=version, local_dir=folder, ignore_patterns=SKIP)

    files = [p for p in folder.rglob("*") if p.is_file() and ".cache" not in p.parts]
    names = {p.name for p in files}
    if "model.safetensors" in names:
        files = [p for p in files if p.name != "pytorch_model.bin"]

    prefix = f"{stage}/{name}/{version}"
    for old in client.list_objects(BUCKET, prefix=prefix + "/", recursive=True):
        client.remove_object(BUCKET, old.object_name)
    for path in files:
        client.fput_object(BUCKET, f"{prefix}/{path.relative_to(folder)}", str(path))

    if stage == "embedding":
        single, two_hop = test_f1(name, "tinyroberta-squad2")
    else:
        single, two_hop = test_f1("all-MiniLM-L6-v2", name)

    entry = {
        "stage": stage,
        "name": name,
        "version": version,
        "source": f"https://huggingface.co/{repo}",
        "size_on_disk_mb": round(sum(p.stat().st_size for p in files) / 1e6, 1),
        "test_f1_single_pass": single,
        "test_f1_two_hop": two_hop,
    }
    metadata = json.dumps(entry, indent=2).encode()
    client.put_object(BUCKET, f"{prefix}/metadata.json",
                      io.BytesIO(metadata), len(metadata))
    registry.append(entry)

table = pd.DataFrame(registry)
table.to_csv(RESULTS_DIR / "registry.csv", index=False)
print(table.to_string(index=False))

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

pytorch_model.bin:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

data_config.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

train_script.py: 0.00B [00:00, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

... (25 outputs left out)

pytorch_model.bin:   0%|          | 0.00/261M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes:   0%|          | 0.00/399 [00:00<?, ?B/s]

    stage                                                  name                                  version                                                                                   source  size_on_disk_mb  test_f1_single_pass  test_f1_two_hop
embedding                                      all-MiniLM-L6-v2 1110a243fdf4706b3f48f1d95db1a4f5529b4d41                            https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2             91.6                0.423            0.460
embedding                                     all-MiniLM-L12-v2 a50ef00143b4d5391434df20ae11632588ac25be                           https://huggingface.co/sentence-transformers/all-MiniLM-L12-v2            134.2                0.423            0.469
embedding                                     all-mpnet-base-v2 e8c3b32edf5434bc2275fc9bab85f82640a19130                           https://huggingface.co/sentence-transformers/all-mpnet-base-v2            438.7                0.446            0.490
embe